# House Prices

## Setup And Load Best Model

In [53]:
import numpy as np
import pandas as pd
import mlflow
import dagshub
import warnings
warnings.filterwarnings("ignore")

import mlflow.xgboost
from scipy.stats import skew

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='house-prices', mlflow=True)
model = mlflow.xgboost.load_model('models:/best_house_price_model/latest')
feature_names = model.get_booster().feature_names

Initialized MLflow to track repo "LukaBatilashvili07/house-prices"

Repository LukaBatilashvili07/house-prices initialized!

## Load Data

In [54]:
train_data = pd.read_csv('data/train.csv')
test_data = pd.read_csv('data/test.csv')
test_ids = test_data['Id']
train_cleaned = train_data.copy()
train_cleaned = train_cleaned.drop(train_cleaned[(train_cleaned['GrLivArea'] > 4000) & (train_cleaned['SalePrice'] < 300000)].index).reset_index(drop=True)
all_data = pd.concat([train_cleaned.drop(['SalePrice'], axis=1), test_data]).reset_index(drop=True)
all_data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


## Data Cleaning

In [55]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 
             'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']

for col in none_cols:
    all_data[col] = all_data[col].fillna('None')


zero_cols = ['GarageYrBlt', 'GarageCars', 'MasVnrArea', 'GarageArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF'
             , 'BsmtFullBath', 'BsmtHalfBath']

for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType', 'Functional', 'Utilities']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

## Feature Engineering

In [56]:
def feature_engineering(df):
    df = df.copy()
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
    df['Age'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)

    qual_mapping = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
    qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
                'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
    for col in qual_cols:
        if col in df.columns:
            df[col] = df[col].map(qual_mapping).fillna(0)

    categorical_cols = df.select_dtypes(include=['object']).columns
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    return df

all_data_fe = feature_engineering(all_data)
all_data_fe.head()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,60,65.0,8450,7,5,2003,2003,196.0,4,...,False,False,False,False,True,False,False,False,True,False
1,2,20,80.0,9600,6,8,1976,1976,0.0,3,...,False,False,False,False,True,False,False,False,True,False
2,3,60,68.0,11250,7,5,2001,2002,162.0,4,...,False,False,False,False,True,False,False,False,True,False
3,4,70,60.0,9550,7,5,1915,1970,0.0,3,...,False,False,False,False,True,False,False,False,False,False
4,5,60,84.0,14260,8,5,2000,2000,350.0,4,...,False,False,False,False,True,False,False,False,True,False


In [57]:
n_train = train_cleaned.shape[0]

X_test_full = all_data_fe[n_train:].drop(['Id'], axis=1, errors='ignore')

num_features = X_test_full.select_dtypes(include=['number']).columns
skewed = X_test_full[num_features].apply(lambda x: skew(x.dropna()))
skewed_features = skewed[skewed > 0.75].index
X_test_full[skewed_features] = np.log1p(X_test_full[skewed_features].clip(lower=0))
X_test = X_test_full.reindex(columns=feature_names, fill_value=0)

## Prediction

In [58]:
log_preds = model.predict(X_test)
preds = np.expm1(log_preds)
print(f'Min: {preds.min()}, Max: {preds.max()}, Mean: {preds.mean()}')

Min: 78995.796875, Max: 513111.59375, Mean: 199915.03125


## Submission

In [59]:
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,Id,SalePrice
0,1461,152847.203125
1,1462,187530.593750
2,1463,234751.312500
3,1464,212288.062500
4,1465,182481.828125
